In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
sensitive_scanner.py
掃描指定資料夾內所有文字檔案，用正則表達式(Regex)偵測常見敏感資訊。

使用方式：
    python sensitive_scanner.py <資料夾路徑> [輸出檔案]

範例：
    python sensitive_scanner.py ./data                 # 只印到終端機
    python sensitive_scanner.py ./data report.csv       # 輸出 CSV
    python sensitive_scanner.py ./data report.json      # 輸出 JSON

新增/刪除規則：
    只需要編輯下方 "RULES" 區塊，新增一行 Rule(...)，或刪除/註解掉某一行即可，
    不需要更動其他程式邏輯。
"""

import os
import re
import csv
import json
import sys
from dataclasses import dataclass
from datetime import datetime
from typing import Callable, Optional


# ========================================================================
# 二次驗證函式（可選）— 用來降低誤報，供下方 RULES 的 validator 欄位引用
# ========================================================================

def luhn_valid(value: str) -> bool:
    """信用卡卡號 Luhn 演算法驗證"""
    digits = [int(d) for d in value if d.isdigit()]
    if not (13 <= len(digits) <= 19):
        return False
    checksum = 0
    parity = len(digits) % 2
    for i, d in enumerate(digits):
        if i % 2 == parity:
            d *= 2
            if d > 9:
                d -= 9
        checksum += d
    return checksum % 10 == 0


_TW_ID_LETTER_VALUE = {
    'A': 10, 'B': 11, 'C': 12, 'D': 13, 'E': 14, 'F': 15, 'G': 16,
    'H': 17, 'I': 34, 'J': 18, 'K': 19, 'L': 20, 'M': 21, 'N': 22,
    'O': 35, 'P': 23, 'Q': 24, 'R': 25, 'S': 26, 'T': 27, 'U': 28,
    'V': 29, 'W': 32, 'X': 30, 'Y': 31, 'Z': 33,
}
_TW_ID_WEIGHTS = [1, 9, 8, 7, 6, 5, 4, 3, 2, 1, 1]


def tw_id_valid(value: str) -> bool:
    """台灣身分證字號檢查碼驗證"""
    if len(value) != 10:
        return False
    letter = value[0].upper()
    if letter not in _TW_ID_LETTER_VALUE or value[1] not in ('1', '2'):
        return False
    if not value[2:].isdigit():
        return False
    letter_val = _TW_ID_LETTER_VALUE[letter]
    digits = [letter_val // 10, letter_val % 10] + [int(c) for c in value[1:]]
    return sum(d * w for d, w in zip(digits, _TW_ID_WEIGHTS)) % 10 == 0


# ========================================================================
# 規則定義區塊 — 想增加/刪除偵測項目，只要在這個 list 裡新增或刪除一行
# ========================================================================

@dataclass
class Rule:
    name: str                                   # 顯示用的類別名稱
    pattern: str                                 # 正則表達式字串
    flags: int = 0                               # re flags，如 re.IGNORECASE
    validator: Optional[Callable[[str], bool]] = None  # 二次驗證函式（可選）

# 不掃描的資料夾名稱（比對「資料夾名稱」本身，不是完整路徑）
EXCLUDE_DIRS = {
    ".git", ".svn", ".hg",
    "node_modules", "__pycache__", ".venv", "venv", "env",
    ".idea", ".vscode",
    "dist", "build","target","download-picture"
}

# 不掃描的路徑關鍵字（比對「完整路徑」，可排除較特定的子路徑）
EXCLUDE_PATH_KEYWORDS = {
    # "tests/fixtures",
}

RULES = [
    Rule("JWT", r"\beyJ[A-Za-z0-9_-]+\.[A-Za-z0-9_-]+\.[A-Za-z0-9_-]+\b"),
    Rule("API Key - OpenAI", r"\bsk-[A-Za-z0-9]{20,}\b"),
    Rule("API Key - AWS Access Key", r"\b(?:AKIA|ASIA)[0-9A-Z]{16}\b"),
    Rule("API Key - AWS Secret Key",
         r"aws_secret_access_key\s*[:=]\s*['\"]?[A-Za-z0-9/+=]{40}['\"]?",
         flags=re.IGNORECASE),
    Rule("API Key - GitHub", r"\b(?:ghp|gho|ghu|ghs|ghr|github_pat)_[A-Za-z0-9_]{20,}\b"),
    Rule("API Key - Google", r"\bAIza[0-9A-Za-z\-_]{35}\b"),
    Rule("Name","邢立承"),
    Rule("學號","A1125515"),
    Rule("N","n站"),
    # Rule("SHA256 Hash", r"\b[A-Fa-f0-9]{64}\b"),
    # Rule("SHA1 Hash", r"\b[A-Fa-f0-9]{40}\b"),
    # Rule("Email", r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b"),
    # Rule("URL", r"\b(?:https?|ftp)://[^\s\"'<>]+"),
    # Rule("MAC Address", r"\b[0-9A-Fa-f]{2}([:-])[0-9A-Fa-f]{2}(?:\1[0-9A-Fa-f]{2}){4}\b"),
    # Rule("IPv6",
    #      r"\b(?:[A-Fa-f0-9]{1,4}:){7}[A-Fa-f0-9]{1,4}\b"
    #      r"|\b(?:[A-Fa-f0-9]{1,4}:){1,7}:(?:[A-Fa-f0-9]{1,4}:){0,6}[A-Fa-f0-9]{1,4}\b"),
    # Rule("IPv4",
    #      r"\b(?:(?:25[0-5]|2[0-4][0-9]|1[0-9]{2}|[1-9]?[0-9])\.){3}"
    #      r"(?:25[0-5]|2[0-4][0-9]|1[0-9]{2}|[1-9]?[0-9])\b"),
    Rule("台灣身分證字號", r"\b[A-Z][12]\d{8}\b", validator=tw_id_valid),
    # Rule("信用卡卡號", r"\b(?:\d[ -]?){13,19}\b", validator=luhn_valid),
    Rule("電話號碼",
         r"\b09\d{2}[- ]?\d{3}[- ]?\d{3}\b"  ),      # 手機

    # 範例：要新增規則，直接照格式加一行即可，例如：
    # Rule("護照號碼", r"\b[0-9]{9}\b"),
]

# 掃描哪些副檔名的檔案（要調整涵蓋範圍就改這裡）
TARGET_EXTENSIONS = {
    ".txt", ".log", ".csv", ".json", ".xml", ".yml", ".yaml",
    ".md", ".py", ".js", ".ts", ".java", ".c", ".cpp", ".h",
    ".ini", ".conf", ".env", ".sql", ".html", ".htm", ".sh",".ipynb"
}

# 預先編譯所有規則，效能較好
_COMPILED_RULES = [(r.name, re.compile(r.pattern, r.flags), r.validator) for r in RULES]


# ========================================================================
# 掃描邏輯
# ========================================================================

def iter_files(root: str):
    for dirpath, dirnames, filenames in os.walk(root):
        # 就地修改 dirnames，讓 os.walk 直接跳過這些資料夾、不遞迴進去
        dirnames[:] = [d for d in dirnames if d not in EXCLUDE_DIRS]
 
        # 用完整路徑關鍵字再排除一次（可排除比較特定的巢狀路徑）
        norm_dirpath = dirpath.replace(os.sep, "/")
        if any(keyword in norm_dirpath for keyword in EXCLUDE_PATH_KEYWORDS):
            continue
 
        for fname in filenames:
            if os.path.splitext(fname)[1].lower() in TARGET_EXTENSIONS:
                yield os.path.join(dirpath, fname)


def scan_file(filepath: str):
    results = []
    try:
        with open(filepath, "r", encoding="utf-8", errors="ignore") as f:
            lines = f.readlines()
    except (OSError, IsADirectoryError, PermissionError):
        return results

    for lineno, line in enumerate(lines, start=1):
        for name, pattern, validator in _COMPILED_RULES:
            for m in pattern.finditer(line):
                value = m.group(0).strip()
                check_value = re.sub(r"[ -]", "", value) if name == "信用卡卡號" else value
                if validator is None or validator(check_value):
                    results.append((name, value, lineno))
    return results

def print_progress(current: int, total: int, bar_length: int = 30):
    """在同一行即時更新文字進度條"""
    if total == 0:
        return
    ratio = current / total
    filled = int(bar_length * ratio)
    bar = "█" * filled + "░" * (bar_length - filled)
    sys.stdout.write(f"\r掃描進度 |{bar}| {current}/{total} ({ratio * 100:5.1f}%)")
    sys.stdout.flush()
    if current == total:
        sys.stdout.write("\n")
        
def scan_folder(root: str):
    files = list(iter_files(root))          # 先取得完整檔案清單，才能算出總數給進度條用
    total = len(files)
    all_findings = []
 
    for idx, filepath in enumerate(files, start=1):
        for name, value, lineno in scan_file(filepath):
            all_findings.append({"file": filepath, "line": lineno, "category": name, "value": value})
        print_progress(idx, total)
 
    # 進度條跑完後，再統一列出所有命中結果（避免跟進度條的 \r 更新互相干擾畫面）
    if all_findings:
        print()
        for item in all_findings:
            print(f"[{item['file']}] 行 {item['line']:>5} | {item['category']:<24} | {item['value']}")
 
    return all_findings, total


# ========================================================================
# 輸出報表
# ========================================================================

def write_csv(findings, path):
    with open(path, "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=["file", "line", "category", "value"])
        writer.writeheader()
        writer.writerows(findings)


def write_json(findings, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(findings, f, ensure_ascii=False, indent=2)


def print_summary(findings, file_count, elapsed):
    print("\n" + "=" * 60)
    print("掃描摘要")
    print("=" * 60)
    print(f"已掃描檔案數：{file_count}")
    print(f"命中總數    ：{len(findings)}")
    print(f"耗時        ：{elapsed:.2f} 秒")
    if findings:
        by_category = {}
        for item in findings:
            by_category[item["category"]] = by_category.get(item["category"], 0) + 1
        print("\n各類別命中數：")
        for name, _pattern, _validator in _COMPILED_RULES:
            if name in by_category:
                print(f"  {name:<24}: {by_category[name]}")
    print("=" * 60)


# ========================================================================
# 主程式：只需要輸入資料夾路徑，以及可選的輸出檔案路徑
# ========================================================================

def main():
    folder=input("請輸入要掃描的資料夾路徑：").strip()

    print(f"開始掃描：{os.path.abspath(folder)}")
    print(f"掃描時間：{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

    start = datetime.now()
    findings, file_count = scan_folder(folder)
    elapsed = (datetime.now() - start).total_seconds()

    print_summary(findings, file_count, elapsed)


if __name__ == "__main__":
    main()